# Decode Failure Analysis

This notebook investigates why some checkpoints have a lower conversion rate than the remote `baseline_remote` model.

It does not rely on the aggregate evaluator only. Instead, it:

1. Generates samples for a chosen set of models and qubit counts.
2. Decodes each sample individually.
3. Captures the exact failure stage (`tokenizer` or `backend`) and exception.
4. Adds lightweight structural diagnostics for suspicious tensor columns.

This should help identify whether failures come mainly from malformed token columns, invalid gate arity, bad padding behavior, or something else.

In [ ]:
import ast
import os
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path

import hydra
import numpy as np
import pandas as pd
import torch
from hydra.core.global_hydra import GlobalHydra
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from quantum_diffusion.evaluation.evaluator import SRVEvaluator
from my_genQC.inference.sampling import generate_tensors as _generate_tensors

In [ ]:
# Edit only this cell.

DATASET_BASE = "./datasets/paper_quditkit"
QUBIT_COUNTS = [5, 6, 7, 8]
SEED = 1234
SAMPLES_PER_BUCKET = 100
USE_STRATIFIED = True
MAX_FAILURE_ROWS_PER_MODEL_QUBIT = 25

MODEL_SPECS = [
    {
        "label": "baseline_remote",
        "model_dir": None,
        "hf_repo": "Floki00/qc_srv_3to8qubit",
    },
    {
        "label": "baseline",
        "model_dir": "./models/trained/paper_srv_bucket",
        "hf_repo": None,
    },
    {
        "label": "cloob_rn50",
        "model_dir": "./models/trained/cloob_srv_bucket",
        "hf_repo": None,
    },
]

In [ ]:
def _build_cfg(dataset_path, model_dir, hf_repo, num_samples):
    GlobalHydra.instance().clear()
    with hydra.initialize(version_base=None, config_path=str(PROJECT_ROOT / "conf")):
        cfg = hydra.compose(config_name="config.yaml", overrides=["evaluation=paper_srv"])
    cfg = cfg["evaluation"]
    cfg.dataset = str(Path(dataset_path).expanduser().resolve())
    cfg.model_dir = str(Path(model_dir).expanduser().resolve()) if model_dir else None
    cfg.hf_repo = hf_repo
    cfg.num_samples = int(num_samples)
    cfg.max_gates = 16
    cfg.save_output = False
    cfg.wandb.enable = False
    return cfg


def _stratified_indices(dataset, samples_per_bucket, seed):
    rng = random.Random(seed)
    bucket_indices = defaultdict(list)
    for i, label in enumerate(dataset.y):
        text = str(label)
        srv = ast.literal_eval(text[text.find("["):text.find("]") + 1])
        n_entangled = sum(1 for v in srv if v == 2)
        bucket_indices[n_entangled].append(i)

    idx = []
    for bucket in sorted(bucket_indices):
        pool = bucket_indices[bucket]
        idx.extend(rng.sample(pool, min(samples_per_bucket, len(pool))))
    return idx


def _instruction_preview(instructions, max_items=8):
    out = []
    for ins in instructions.data[:max_items]:
        out.append(
            {
                "name": ins.name,
                "controls": list(ins.control_nodes),
                "targets": list(ins.target_nodes),
            }
        )
    return out


def _analyze_tensor_columns(tensor, valid_gate_ids, pad_constant):
    suspicious = []
    for t in range(tensor.shape[1]):
        col = tensor[:, t].detach().cpu()
        values = col.tolist()

        non_pad = [int(v) for v in values if int(v) != int(pad_constant)]
        non_zero = [v for v in non_pad if v != 0]
        if not non_zero:
            continue

        abs_ids = sorted({abs(v) for v in non_zero})
        pos = [i for i, v in enumerate(values) if v > 0 and v != pad_constant]
        neg = [i for i, v in enumerate(values) if v < 0]

        reasons = []
        if len(abs_ids) > 1:
            reasons.append("mixed_gate_ids")
        if any(g not in valid_gate_ids for g in abs_ids):
            reasons.append("unknown_gate_id")
        if len(neg) > 0 and len(pos) == 0:
            reasons.append("control_without_target")
        if len(pos) > 1:
            reasons.append("multiple_targets")
        if len(neg) > 1:
            reasons.append("multiple_controls")

        if reasons:
            suspicious.append(
                {
                    "t": t,
                    "values": values,
                    "abs_ids": abs_ids,
                    "num_pos": len(pos),
                    "num_neg": len(neg),
                    "reasons": reasons,
                }
            )

    return suspicious


def _decode_one(evaluator, tensor, prompt, sample_idx, qubit_count, model_label):
    tensor_cpu = tensor.detach().cpu()
    valid_gate_ids = set(evaluator.tokenizer.vocabulary_inverse.keys())
    pad_constant = len(evaluator.dataset.gate_pool) + 1
    suspicious_cols = _analyze_tensor_columns(tensor_cpu, valid_gate_ids, pad_constant)

    try:
        instructions = evaluator.tokenizer.decode(tensor_cpu)
    except Exception as exc:
        return {
            "model": model_label,
            "num_qubits": qubit_count,
            "sample_idx": sample_idx,
            "prompt": prompt,
            "success": False,
            "failure_stage": "tokenizer",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "instruction_preview": None,
            "suspicious_cols": suspicious_cols,
            "tensor_head": tensor_cpu[:, :16].tolist(),
        }

    try:
        backend_obj = evaluator.simulator.backend.genqc_to_backend(instructions, place_barriers=False)
        return {
            "model": model_label,
            "num_qubits": qubit_count,
            "sample_idx": sample_idx,
            "prompt": prompt,
            "success": True,
            "failure_stage": None,
            "error_type": None,
            "error_message": None,
            "instruction_preview": _instruction_preview(instructions),
            "suspicious_cols": suspicious_cols,
            "tensor_head": tensor_cpu[:, :16].tolist(),
            "backend_obj": backend_obj,
        }
    except Exception as exc:
        return {
            "model": model_label,
            "num_qubits": qubit_count,
            "sample_idx": sample_idx,
            "prompt": prompt,
            "success": False,
            "failure_stage": "backend",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "instruction_preview": _instruction_preview(instructions),
            "suspicious_cols": suspicious_cols,
            "tensor_head": tensor_cpu[:, :16].tolist(),
        }


def inspect_model_qubit(model_spec, qubit_count, dataset_base, samples_per_bucket, use_stratified, seed):
    dataset_path = f"{dataset_base}/srv_{qubit_count}q_dataset"
    requested_samples = samples_per_bucket * qubit_count

    cfg = _build_cfg(dataset_path, model_spec.get("model_dir"), model_spec.get("hf_repo"), requested_samples)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    evaluator = SRVEvaluator(config=cfg)

    if use_stratified:
        idx = _stratified_indices(evaluator.dataset, samples_per_bucket, seed)
        evaluator.samples = len(idx)
        evaluator.idx = idx
        prompts = [str(evaluator.dataset.y[i]) for i in idx]
    else:
        evaluator.samples = min(requested_samples, len(evaluator.dataset.y))
        evaluator.idx = random.sample(range(len(evaluator.dataset.y)), k=evaluator.samples)
        prompts = [str(evaluator.dataset.y[i]) for i in evaluator.idx]

    tensors_out = _generate_tensors(
        pipeline=evaluator.pipeline,
        prompt=prompts,
        samples=evaluator.samples,
        system_size=evaluator.system_size,
        num_of_qubits=evaluator.num_qubits,
        max_gates=evaluator.max_gates,
        g=cfg.model_params.guidance_scale,
        auto_batch_size=cfg.model_params.auto_batch_size,
        enable_params=False,
        no_bar=False,
    )

    rows = []
    valid_backend = []
    valid_indices = []
    for sample_idx, (tensor, prompt) in enumerate(zip(tensors_out, prompts)):
        row = _decode_one(
            evaluator=evaluator,
            tensor=tensor,
            prompt=prompt,
            sample_idx=sample_idx,
            qubit_count=qubit_count,
            model_label=model_spec["label"],
        )
        rows.append(row)
        if row["success"]:
            valid_backend.append(row.pop("backend_obj"))
            valid_indices.append(sample_idx)

    err_rows = [row for row in rows if not row["success"]]
    stage_counter = Counter(row["failure_stage"] for row in err_rows)
    error_counter = Counter((row["failure_stage"], row["error_type"], row["error_message"]) for row in err_rows)

    summary = {
        "model": model_spec["label"],
        "num_qubits": qubit_count,
        "samples": len(rows),
        "valid": len(rows) - len(err_rows),
        "invalid": len(err_rows),
        "conversion_rate": (len(rows) - len(err_rows)) / len(rows) if rows else 0.0,
        "tokenizer_failures": stage_counter.get("tokenizer", 0),
        "backend_failures": stage_counter.get("backend", 0),
        "top_errors": error_counter.most_common(10),
    }

    return summary, rows


def errors_to_frame(rows, keep_only_failures=True, max_rows=None):
    filtered = [row for row in rows if (not keep_only_failures) or (not row["success"])]
    if max_rows is not None:
        filtered = filtered[:max_rows]

    slim = []
    for row in filtered:
        suspicious_reason_counter = Counter(
            reason
            for col in row["suspicious_cols"]
            for reason in col["reasons"]
        )
        slim.append(
            {
                "model": row["model"],
                "num_qubits": row["num_qubits"],
                "sample_idx": row["sample_idx"],
                "failure_stage": row["failure_stage"],
                "error_type": row["error_type"],
                "error_message": row["error_message"],
                "num_suspicious_cols": len(row["suspicious_cols"]),
                "suspicious_reasons": dict(suspicious_reason_counter),
                "instruction_preview": row["instruction_preview"],
                "prompt": row["prompt"],
            }
        )

    return pd.DataFrame(slim)

In [ ]:
summaries = []
all_rows = {}

for spec in MODEL_SPECS:
    print(f"\n=== {spec['label']} ===")
    for q in QUBIT_COUNTS:
        summary, rows = inspect_model_qubit(
            model_spec=spec,
            qubit_count=q,
            dataset_base=DATASET_BASE,
            samples_per_bucket=SAMPLES_PER_BUCKET,
            use_stratified=USE_STRATIFIED,
            seed=SEED,
        )
        summaries.append(summary)
        all_rows[(spec['label'], q)] = rows
        print(
            f"  {q}q  samples={summary['samples']}  valid={summary['valid']}  invalid={summary['invalid']}  "
            f"conversion={summary['conversion_rate']:.4f}  tokenizer={summary['tokenizer_failures']}  backend={summary['backend_failures']}"
        )

In [ ]:
summary_df = pd.DataFrame(summaries)
display(summary_df[[
    'model', 'num_qubits', 'samples', 'valid', 'invalid', 'conversion_rate', 'tokenizer_failures', 'backend_failures'
]])

In [ ]:
for summary in summaries:
    key = (summary['model'], summary['num_qubits'])
    print(f"\n=== Top errors: {summary['model']} / {summary['num_qubits']}q ===")
    for (stage, err_type, err_msg), count in summary['top_errors'][:10]:
        print(f"[{count:4d}] {stage} | {err_type} | {err_msg}")

In [ ]:
# Pick one model/qubit pair to inspect failure rows in detail.
INSPECT_MODEL = 'baseline'
INSPECT_QUBITS = 8

detail_df = errors_to_frame(
    all_rows[(INSPECT_MODEL, INSPECT_QUBITS)],
    keep_only_failures=True,
    max_rows=MAX_FAILURE_ROWS_PER_MODEL_QUBIT,
)
display(detail_df)